# Volatility Regression 04 — freeze the confirmation specification

Pre-registered in VR 011 (Ivan, 2026-10-09). This notebook loads **only expiries ≤ 2026-10-08**, fits B0 and B0_ATM **once** per asset and target, and writes `04_frozen_spec.json` (coefficients, centring/scaling, ε, train mean, admitted features, protocol). Notebook 05 scores later sessions with these frozen numbers.

- It refuses to overwrite an existing frozen spec.
- It requires the rebuilt feature table (time-series schema 0.6.0, with `mid_path_*` columns). Rebuild features first.
- Commit `04_frozen_spec.json` **before** running 05.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "src" / "data_engineering").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Launch from the repository root or one of its subfolders.")
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
DATA_ROOT = os.getenv("DERIBIT_DATA_ROOT")
if not DATA_ROOT:
    raise RuntimeError("DERIBIT_DATA_ROOT must be set in .env.")
DB_PATH = Path(DATA_ROOT) / "database" / "research.duckdb"
if not DB_PATH.is_file():
    raise FileNotFoundError(DB_PATH)
from data_engineering.modeling import vol_ols_confirm as vc
OUTPUT = ROOT / "notebooks" / "Volatility_Regression"
SPEC_PATH = OUTPUT / "04_frozen_spec.json"
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(ROOT / "tests"),
                "-p", "test_vol_ols_confirm.py", "-v"], cwd=ROOT, check=True)

In [ ]:
if SPEC_PATH.exists():
    raise RuntimeError(f"{SPEC_PATH.name} already exists. The specification is frozen; "
                       "do not refit. A change requires a new version and a later window.")
with duckdb.connect(str(DB_PATH), read_only=True) as con:
    con.execute("SET TimeZone = 'UTC'")
    cols = set(con.execute("DESCRIBE time_series_feature_state").df()["column_name"])
    missing = [c for c in vc.MID_COLUMNS if c not in cols]
    if missing:
        raise RuntimeError(f"Feature table lacks {missing}. Rebuild features (schema 0.6.0) first.")
    df = con.execute("""
        SELECT * FROM time_series_feature_state
        WHERE TRY_CAST(expiry_date AS DATE) <= CAST(? AS DATE)
        ORDER BY underlying, state_time
    """, [vc.TRAIN_LAST_EXPIRY]).df()
ASSETS = tuple(sorted(df["underlying"].dropna().astype(str).unique()))
print("Training rows:", len(df), "| assets:", ASSETS)
print("Span:", df.state_time.min(), "to", df.state_time.max())

In [ ]:
specs, summary, errors = {}, [], []
for asset in ASSETS:
    meta, X = vc.panel(df[df["underlying"] == asset])
    specs[asset] = {}
    for target in (vc.PRIMARY_TARGET, *vc.SECONDARY_TARGETS):
        try:
            s = vc.freeze_asset(meta, X, target)
            specs[asset][target] = s
            for model, rec in s["models"].items():
                summary.append({"underlying": asset, "target": target, "model": model,
                                "fitted": rec["fitted"], "n_train": rec["n_train"],
                                "p": len(rec["features_used"]),
                                "not_admitted": ",".join(sorted(set(rec["features_requested"])
                                                                - set(rec["features_admitted"]))),
                                "train_sessions": len(s["train_sessions"]),
                                "reason": rec.get("reason", "")})
        except Exception as exc:
            errors.append({"underlying": asset, "target": target,
                           "reason": f"{type(exc).__name__}: {exc}"})
summary_df = pd.DataFrame(summary)
errors_df = pd.DataFrame(errors)
display(summary_df)
if not errors_df.empty:
    display(errors_df)
primary_ok = all(specs.get(a, {}).get(vc.PRIMARY_TARGET, {}).get("models", {}).get(m, {}).get("fitted")
                 for a in vc.PRIMARY_ASSETS for m in vc.MODELS)
print("All primary asset/model fits succeeded:", primary_ok)

In [ ]:
frozen = {
    "registration": vc.registration(),
    "frozen_at_utc": datetime.now(timezone.utc).isoformat(),
    "training_data": {"rows": int(len(df)), "first_state": str(df.state_time.min()),
                      "last_state": str(df.state_time.max())},
    "primary_fits_ok": bool(primary_ok),
    "specs": specs,
    "errors": errors,
}
SPEC_PATH.write_text(vc.dumps(frozen), encoding="utf-8")
summary_df.to_csv(OUTPUT / "04_freeze_summary.csv", index=False)
print("Wrote", SPEC_PATH.name, "— commit it before running notebook 05.")